# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [72]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [65]:
import os, sys
from langchain_community.document_loaders import PyPDFLoader

# print (PyPDFLoader.__doc__) # check exists.

# made docs folder
from pathlib import Path
path = os.getcwd()
folder = 'doc'
# print (path + '/' +folder)
new_directory_path = Path(path + '/' + folder)

if not os.path.exists (folder):
    new_directory_path.mkdir(parents=True, exist_ok=True)
    print (f"folder made")

# get doc from online 

## Select a Document


from urllib.request import urlretrieve

url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
filename = "ai_report_2025.pdf"
path = path + '/' + folder
fullpath = path + '/' + filename
print (fullpath)

def download_pdf_urllib(url, fullpath):

    try:
        urlretrieve(url, fullpath)
        print(f"Successfully downloaded '{filename}' using urllib")
    except Exception as e:
        print(f"An error occurred: {e}")

download_pdf_urllib(url, fullpath)


# now get the doc length
from langchain_community.document_loaders import PyPDFLoader

# file_path = "../example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(fullpath)

docs = loader.load()

print(len(docs))

# now get content
print(docs[0].metadata)

if docs:
    metadata = docs[0].metadata
    #print(first_page_metadata)
    # The author is usually under 'author' or 'creators'
    author = metadata.get('author')
    title = metadata.get('title')
    print(f"Author: {author}")
    print(f"Title: {title}")



/Users/darko-adm/work/dsi/deploying-ai/02_activities/doc/ai_report_2025.pdf
Successfully downloaded 'ai_report_2025.pdf' using urllib
26
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': '/Users/darko-adm/work/dsi/deploying-ai/02_activities/doc/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}
Author: Aditya Challapally
Title: None


In [73]:

# now get the doc length
from langchain_community.document_loaders import PyPDFLoader

# file_path = "../example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(fullpath)

docs = loader.load()

print(len(docs))

# now get content
print(docs[0].metadata)

if docs:
    metadata = docs[0].metadata
    #print(first_page_metadata)
    # The author is usually under 'author' or 'creators'
    author = metadata.get('author')
    title = metadata.get('title')
    print(f"Author: {author}")
    print(f"Title: {title}")

    # Grab the first page
first_page_text = docs[0].page_content

# The title is usually the first non-empty line
title_fallback = first_page_text.strip().split('\n')[3]
print(f"Likely Title: {title_fallback}")


#print(list(docs[0].metadata))

26
{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-13T21:18:19-07:00', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_siteid': '72f988bf-86f1-41af-91ab-2d7cd011db47', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_method': 'Privileged', 'msip_label_87867195-f2b8-4ac2-b0b6-6bb73cb33afc_enabled': 'True', 'author': 'Aditya Challapally', 'moddate': '2025-07-13T21:18:19-07:00', 'source': '/Users/darko-adm/work/dsi/deploying-ai/02_activities/doc/ai_report_2025.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}
Author: Aditya Challapally
Title: None
Likely Title: The GenAI Divide  


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [74]:
from openai import OpenAI
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

models = client.models.list()

#for model in models:
#   print (model)


# add the text of my pdf doc. chunk it into smaller sections.
paragraphs = [p.strip() for p in full_text.split("\n\n") if p.strip()]

#messages = [{"role": "user", "content": p} for p in paragraphs]
#print (messages)
# chunked_text = "\n\n".join(paragraphs)
#print (chunked_text)
#raw_text = "\n\n".join(d.page_content for d in docs)
#paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
#full_doc = "\n\n".join(paragraphs)
full_doc = "\n\n".join(d.page_content for d in docs)


# get summary of document.
#for i, para in enumerate(paragraphs):
#    response = client.responses.create(
##        model="gpt-4o",
#        input=[{"role": "user", "content": summary_prompt1 + "\n\n" + full_doc}]
#    )

total_input_tokens = 0
total_output_tokens = 0

title_page = first_page_text.strip().split('\n')[3]


author_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Analyze the included document and can you tell me know the author or authors are of this document? "

response = client.responses.create(
    model="gpt-4o",
    input=[
          {"role": "system", "content": author}, 
        {"role": "user", "content": title_page}], max_output_tokens=500
)

author = response.output_text
usage = response.usage
#print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens

title_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Analyze the included document and can you tell me what is the exact title of this document? "

response = client.responses.create(
    model="gpt-4o",
    input=[
          {"role": "system", "content": title_prompt}, 
        {"role": "user", "content": title_page}], max_output_tokens=500
)

title = response.output_text
#print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens




relevance_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Analyze the included document and explain to me why this document is important and relevant " \
    "for an AI professional in their professional development. " \
    "Provide your summary in one paragraph length."

response = client.responses.create(
    model="gpt-4o",
    input=[
          {"role": "system", "content": relevance_prompt}, 
        {"role": "user", "content": full_doc}], max_output_tokens=500
)

usage = response.usage

relevance = response.output_text
#print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens

user_content = full_doc
summary_prompt = "You are a professional guidance counselor at a university with over 10 years experience at the job. " \
    "Summarize the following document using the tonal format of African-American Vernacular English. "

response = client.responses.create(
    model="gpt-4o", 
    input=[
        {"role": "system", "content": summary_prompt}, 
        {"role": "user", "content": full_doc}], max_output_tokens=1000
)

usage = response.usage

summary = response.output_text
#print (usage.output_tokens)
total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens


tone_prompt = "Explin to me in detail the tone used in crafting this prompt." 

response = client.responses.create(
    model="gpt-4o", 
    input=[
        {"role": "user", "content": tone_prompt},
        {"role": "user", "content": full_doc}], max_output_tokens=500
)

usage = response.usage

tone = response.output_text

total_input_tokens += usage.input_tokens
total_output_tokens += usage.output_tokens

from IPython.display import display, Markdown
display(Markdown(f"Author: {author}"))
display(Markdown(f"Title: {title}"))
display(Markdown(f"Relevance: {relevance}"))
display(Markdown(f"Summary: {summary}"))
display(Markdown(f"Tone: {tone}"))
display(Markdown(f"Total input tokens: {total_input_tokens}"))
display(Markdown(f"Total output tokens: {total_output_tokens}"))


#response = client.responses.create(
##    model = 'gpt-4o',
#    input = docs
#)

#from IPython.display import display, Markdown
#display(Markdown(response.output_text))

Author: The "GenAI Divide" typically refers to the growing gap between those who have access to and can effectively use generative AI technologies and those who cannot. This divide can manifest in several ways:

1. **Access to Technology:** Some individuals or communities may lack access to the necessary infrastructure, such as high-speed internet and powerful computing resources, to utilize advanced AI tools.

2. **Knowledge and Skills:** There’s a disparity in understanding and skill levels when it comes to using and developing AI technologies. This includes factors such as education, training, and familiarity with digital tools.

3. **Economic Factors:** The cost of implementing and maintaining AI tools can be prohibitive for smaller organizations or underserved communities, widening the gap between larger entities that can afford these technologies and those that cannot.

4. **Geopolitical Disparities:** Different countries may have varying levels of investment and focus on AI development, leading to an international divide in AI capabilities and advancements.

5. **Cultural and Ethical Concerns:** Diverse communities may have different perspectives and ethical considerations regarding the use of AI, which can affect their adoption and integration of the technology.

Addressing the GenAI Divide involves developing inclusive policies, investing in education and training, ensuring equitable access to technology, and fostering collaborative efforts across different sectors and communities.

Title: The exact title of the document is "The GenAI Divide."

Relevance: The document "The GenAI Divide: State of AI in Business 2025" is crucial for an AI professional seeking to understand the current and future landscape of AI integration in organizations. It highlights the challenges and opportunities in crossing the "GenAI Divide," where despite high adoption of generative AI technologies, only a small fraction achieves meaningful transformation due to a lack of learning and integration capabilities. The report underscores the importance of learning-capable systems that retain feedback and adapt to context, which contrasts starkly with the failure of generic tools in delivering substantial business value. For an AI professional, this document serves as an insightful guide on successful strategies for both builders and buyers to navigate the AI landscape, emphasizing the importance of customization, workflow integration, and the emerging concept of an "Agentic Web" where autonomous, adaptive systems operate seamlessly. This understanding is pivotal for AI professionals to innovate, influence decision-making, and lead successful AI initiatives within their organizations.

Summary: Aight, so let me break it down for ya. This here document talkin' 'bout the "GenAI Divide," right? Basically, it's all about how businesses investin' big money in AI, but most ain't seein' no real benefits from it. Outta all these companies playin' with AI, only 5% actually gettin' some real value. They call it a divide ‘cause most folks on one side only playin’ with tools like ChatGPT for simple stuff, but it ain't changin' their whole business game.

They did all kinda research, interviewin' folks, and checkin' out real-world cases. It turns out, the companies that make this AI work are all about customizing stuff specific to their needs and makin' sure these tools learn and adapt over time. It ain't so much about technical skills or fancy tools, it's about how well it fit with what they already do.

Now, a whole lotta folks on the wrong side of this divide are actin' like they about to leap across it, but they get stuck 'cause AI ain't really learnin' or fittin' into what they do day-to-day. On top of that, people are still usin' personal AI tools on the down-low to get their jobs done, while official company AI projects are barely rollin’.

The real players here, the ones that succeed, they ain't buildin' stuff themselves from scratch. They partner up with folks who know what they doin'. And the ones buyin' these tools are treatin' 'em more like serious business partners than just some tech thing they tryna try out.

End of the day, if a company wanna cross that GenAI Divide, they gotta quit messin’ with static tools, partner up right, and focus on how all this AI gonna fit into their daily grind, learnin’ and growin’ with 'em.

Tone: The tone used in crafting this prompt is analytical and professional. It is intended for a knowledgeable audience, likely composed of business leaders, academics, and professionals involved in AI implementation. The language is formal and precise, providing detailed insights and structured arguments while emphasizing data-driven findings. The tone is authoritative, aiming to convey expertise and thorough research, yet it remains neutral and objective, focusing on presenting facts and strategic recommendations. There is also an element of urgency, particularly regarding the closing window to adopt AI effectively.

Total input tokens: 32253

Total output tokens: 1204

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [76]:

# summarization metrics
import os
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

os.environ["OPENAI_API_KEY"] = os.getenv("API_GATEWAY_KEY") or ""

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

metric_answer = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,   
)

test_case_author = LLMTestCase(
    input=title_page,
    actual_output=author)

metric_answer.measure(test_case_author)

evaluate(test_cases=[test_case_author], metrics=[metric_answer])

# evalaute summary
test_case_summary = LLMTestCase(input=summary_prompt, actual_output=summary)

metric_summary = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the model accurately summarize the content of the document that the user inputed? ",
        "Is the summary provided to the user within the constaint of one paragraph length? ",
        "Does the summary abide by the tone of African-American Vernacular English? ",
        "Does a higher score in this eval mean the summary is concise and comprehensive? "
    ]
)

evaluate(test_cases=[test_case_summary], metrics=[metric_summary])



from IPython.display import display, Markdown
display(Markdown(f'**Score**: {metric_answer.score}'))
display(Markdown(f'**Reason**: {metric_answer.reason}'))
display(Markdown(f'**Score**: {metric_summary.score}'))
display(Markdown(f'**Reason**: {metric_summary.reason}'))



Output()

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4o-mini, reason: The score is 1.00 because the response directly addresses the input without any irrelevant statements., error: None)

For test case:

  - input: The GenAI Divide  
  - actual output: The "GenAI Divide" typically refers to the growing gap between those who have access to and can effectively use generative AI technologies and those who cannot. This divide can manifest in several ways:

1. **Access to Technology:** Some individuals or communities may lack access to the necessary infrastructure, such as high-speed internet and powerful computing resources, to utilize advanced AI tools.

2. **Knowledge and Skills:** There’s a disparity in understanding and skill levels when it comes to using and developing AI technologies. This includes factors such as education, training, and familiarity with digital tools.

3. **Economic Factors:** The cost of implementing and maint

✓ Evaluation completed 🎉! (time taken: 7.98s | token cost: 0.00046395 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

IndexError: list index out of range

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
